# 09.10 · 표가 있는 Word 문서로 RAG 만들기

2026-10-15 수업의 `09.10` 화면을 따라 완성한 실습입니다. 기본 흐름은 **문서 읽기 → 표 구조 보존 → 분할 → Upstage 임베딩 → Pinecone 저장 → 검색 → 답변**입니다.

이 노트북은 `C:\ai\Source\07_LLM` 폴더에서 실행합니다. Jupyter 커널은 **Python (llm)** 을 선택하세요. API 요청은 실행할 때만 발생합니다. Pinecone 키는 파일에 저장하지 않고 실행 세션에만 입력합니다.

## 1. 실습 파일과 API 키 확인

`.env`의 설정을 읽습니다. 이 단계는 API를 호출하지 않으므로 키가 없어도 문서와 표를 확인할 수 있습니다. 4단계에서 필요한 키를 숨김 입력창으로 받을 수 있습니다. 입력한 키는 노트북 출력과 파일에 남기지 않습니다.

In [ ]:
import os
from getpass import getpass
from pathlib import Path
from dotenv import load_dotenv

BASE_DIR = Path.cwd()
DOCX_PATH = BASE_DIR / "tax_docx" / "소득세법_with_table.docx"
assert DOCX_PATH.is_file(), f"파일을 찾지 못했습니다: {DOCX_PATH}"
load_dotenv(BASE_DIR / ".env", override=False)
print("문서:", DOCX_PATH.name)

## 2. 수업 방식: 텍스트만 읽었을 때 확인

`Docx2txtLoader`는 본문을 읽기 쉽지만 표의 열 관계가 흐려질 수 있습니다. 아래 코드는 비교용이며 Pinecone에는 올리지 않습니다.

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=100)
flat_documents = Docx2txtLoader(str(DOCX_PATH)).load_and_split(text_splitter=splitter)
print("일반 텍스트 조각 수:", len(flat_documents))
print("세율표 주변 예시:\n", next((d.page_content[:900] for d in flat_documents if "과세표준" in d.page_content and "세율" in d.page_content), "찾지 못함"))

## 3. 개선: 표의 각 행과 열을 함께 보존

Word 파일의 문단과 표를 원래 순서대로 읽습니다. 본문은 1,500자씩 나누고, 세율표는 독립된 문서 한 조각으로 만듭니다. 표 헤더가 각 행에 붙으므로 검색 결과를 읽을 때 금액 구간과 세율의 관계가 남습니다.

In [ ]:
from docx import Document as WordDocument
from docx.table import Table
from langchain_core.documents import Document

word = WordDocument(DOCX_PATH)
source_name = DOCX_PATH.name
documents = []
paragraph_buffer = []
table_count = 0

def add_paragraphs():
    text = "\n".join(paragraph_buffer).strip()
    if text:
        for chunk in splitter.split_text(text):
            documents.append(Document(page_content=chunk, metadata={
                "source": source_name, "content_type": "paragraph"
            }))
    paragraph_buffer.clear()

for item in word.iter_inner_content():
    if isinstance(item, Table):
        add_paragraphs()
        table_count += 1
        rows = [[cell.text.replace("\n", " ").replace("|", "\\|").strip()
                 for cell in row.cells] for row in item.rows]
        if rows:
            header = rows[0]
            lines = [f"[표 {table_count}] {header[0]}",
                     " | ".join(header),
                     " | ".join(["---"] * len(header))]
            lines += [" | ".join(row) for row in rows[1:]]
            documents.append(Document(page_content="\n".join(lines), metadata={
                "source": source_name, "content_type": "table", "table_index": table_count
            }))
    elif item.text.strip():
        paragraph_buffer.append(item.text.strip())
add_paragraphs()

table_documents = [d for d in documents if d.metadata["content_type"] == "table"]
print("본문 조각:", len(documents) - len(table_documents), "| 표:", len(table_documents))
assert table_documents, "표가 추출되지 않았습니다."
assert "과세표준" in table_documents[0].page_content and "세율" in table_documents[0].page_content
print(table_documents[0].page_content)

## 4. Upstage 임베딩과 Pinecone 인덱스 만들기

`UpstageEmbeddings`에는 모델의 기본 이름 `solar-embedding-1-large`를 줍니다. 이 라이브러리가 문서에는 `-passage`, 질문에는 `-query`를 자동으로 붙입니다. 현재 설치된 라이브러리에서 `embedding-query`를 직접 넣으면 `-query-query`가 되어 오류가 나므로 수정했습니다.

인덱스가 이미 있으면 차원을 확인한 뒤 재사용합니다. 다른 차원의 인덱스는 덮어쓰지 않습니다.

In [ ]:
import time
from langchain_upstage import UpstageEmbeddings
from pinecone import Pinecone, ServerlessSpec
from openai import AuthenticationError

if not os.getenv("UPSTAGE_API_KEY"):
    os.environ["UPSTAGE_API_KEY"] = getpass("Upstage API 키 (이 커널에만 저장): ")
embedding = UpstageEmbeddings(model="solar-embedding-1-large")
try:
    dimension = len(embedding.embed_query("과세표준 확인"))
except AuthenticationError:
    os.environ["UPSTAGE_API_KEY"] = getpass("현재 Upstage 키가 무효입니다. 새 API 키 (이 커널에만 저장): ")
    embedding = UpstageEmbeddings(model="solar-embedding-1-large", api_key=os.environ["UPSTAGE_API_KEY"])
    dimension = len(embedding.embed_query("과세표준 확인"))
print("Upstage 임베딩 차원:", dimension)

INDEX_NAME = "tax-index-table"
NAMESPACE = "ch9-10-table-aware-v1"
if not os.getenv("PINECONE_API_KEY"):
    os.environ["PINECONE_API_KEY"] = getpass("Pinecone API 키 (이 커널에만 저장): ")
assert os.getenv("PINECONE_API_KEY"), "Pinecone API 키가 비어 있습니다."
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
existing = set(pc.list_indexes().names())
if INDEX_NAME not in existing:
    pc.create_index(name=INDEX_NAME, dimension=dimension, metric="cosine",
                    spec=ServerlessSpec(cloud="aws", region="us-east-1"))
    for _ in range(60):
        if pc.describe_index(INDEX_NAME).status["ready"]:
            break
        time.sleep(2)
    else:
        raise TimeoutError(f"{INDEX_NAME} 인덱스 준비 대기 시간 초과")
index_description = pc.describe_index(INDEX_NAME)
assert index_description.dimension == dimension, (
    f"기존 인덱스 차원 {index_description.dimension}과 임베딩 차원 {dimension}이 다릅니다."
)
print("Pinecone 인덱스 준비:", INDEX_NAME, "| 차원:", index_description.dimension)

## 5. 문서 저장

같은 문서 조각에는 같은 ID를 부여합니다. 이 셀을 다시 실행해도 같은 ID를 갱신하므로 중복이 쌓이지 않습니다. 수업용 별도 namespace를 사용합니다.

In [ ]:
import hashlib
from langchain_pinecone import PineconeVectorStore

store = PineconeVectorStore(index_name=INDEX_NAME, embedding=embedding,
                            namespace=NAMESPACE, pinecone_api_key=os.environ["PINECONE_API_KEY"])
ids = ["ch9-10-" + hashlib.sha256(
    (str(i) + "\n" + d.page_content).encode("utf-8")
).hexdigest()[:32] for i, d in enumerate(documents)]
written_ids = store.add_documents(documents, ids=ids)
print("저장한 조각 수:", len(written_ids), "| namespace:", NAMESPACE)
assert len(written_ids) == len(documents)

## 6. 검색 결과 확인

먼저 검색 결과의 원문과 출처를 확인합니다. 연봉 5천만 원이라는 정보만으로는 **과세표준**을 알 수 없으므로, 표의 경계를 묻는 질문으로 검색을 점검합니다.

In [ ]:
question = "종합소득 과세표준이 5,000만 원일 때 적용되는 세율 구간과 계산식은 무엇인가요?"
matches = store.similarity_search_with_score(question, k=4)
for rank, (doc, score) in enumerate(matches, 1):
    print(f"\n[{rank}] score={score:.4f} type={doc.metadata.get('content_type')} source={doc.metadata.get('source')}")
    print(doc.page_content[:900])
assert matches, "검색 결과가 없습니다."

## 7. 검색 문맥으로 답변하기

질문 → 검색 → 문맥 정리 → Upstage 모델 → 답변 순서입니다. 근거에 없는 금액은 계산하지 않도록 프롬프트에 명시했습니다. 학습용 예시이며 실제 세금 신고 판단에 사용하지 않습니다.

In [ ]:
from langchain_upstage import ChatUpstage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

retriever = store.as_retriever(search_type="similarity", search_kwargs={"k": 4})
def format_documents(retrieved):
    return "\n\n".join(
        f"[자료 {i} | {d.metadata.get('source')} | {d.metadata.get('content_type')}]\n{d.page_content}"
        for i, d in enumerate(retrieved, 1)
    )

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 제공된 소득세법 문서의 표를 설명하는 학습 도우미입니다. "
     "제공된 자료만 근거로 답하세요. 과세표준과 연봉을 구분하고, 필요한 공제 정보가 없으면 "
     "실제 세액을 계산할 수 없다고 말하세요. 자료가 부족하면 추측하지 마세요. "
     "답변 끝에 근거 자료 번호를 적으세요.\n\n자료:\n{context}"),
    ("human", "{query}"),
])
llm = ChatUpstage(model="solar-pro4", temperature=0)
rag_chain = ({"context": retriever | format_documents,
              "query": RunnablePassthrough()} | prompt | llm | StrOutputParser())
print(rag_chain.invoke(question))

## 8. 스스로 확인할 점

1. 위 검색 결과에 `content_type=table` 조각이 있는가?
2. 표의 `1,400만 원 초과 5,000만 원 이하`와 `5,000만 원 초과`가 구분되는가?
3. 질문을 `연봉 5천만 원의 실제 소득세는?`로 바꾸면, 추가 정보가 필요하다고 답하는가?

API 키는 커널 종료 시 메모리에서 사라집니다. 다음 실행 때 다시 입력하면 됩니다.

# 수업본의 또 다른 표 문서 RAG 흐름 (참고용)

앞의 본편은 Word 표의 행과 열을 보존해 분할하는 실습입니다. 아래는 수업본의 별도 로더·검색·답변 흐름입니다. 같은 인덱스에 중복 저장되는 일을 피하도록 수업본의 코드 셀은 읽기용 코드 블록으로 합쳤습니다. 직접 실행할 때는 백업 원본을 별도 커널에서 열어 필요한 인덱스와 API 키를 확인하세요. 수업본의 저장된 실행 결과는 백업 원본에 남아 있습니다.


### 수업본 코드 셀 1

```python
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:98% !important;}
div.cell.code_cell.rendered{width:98%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li, div.text_cell_render p{width:95% !important;font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
table td, th{font-size:16px;}
table{ margin-left:0 !important;   /* 왼쪽 여백 0 */}
</style>
"""))
```


# 1. knowledge base 구성을 위한 데이터 생성

### 수업본 코드 셀 3

```python
%%time
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader("tax_docx/소득세법_with_table.docx")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,   # 문서를 쪼갤 때 1500글자씩 chunking
    chunk_overlap=20, # 20문자씩 overlap 하여 chunking
    # separators=["\n\n", "\n", " ", ""]
)
# Recursive(재귀적으로) 다음의 순서대로 chunking 시도 : "\n\n", "\n", " ", ""
documents = loader.load_and_split(text_splitter=text_splitter)
len(documents)
```


### 수업본 코드 셀 4

```python
documents[43]
```


### 수업본 코드 셀 5

```python
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv(override=True) # 커널을 시작하지 않고 .env덮어쓰기
embedding = UpstageEmbeddings(
                model = 'embedding-query'
)
```


### 수업본 코드 셀 6

```python
len(embedding.embed_query("소득세법"))
```


### 수업본 코드 셀 7

```python
# pinecone의 index 생성하기
import time
from pinecone import Pinecone, ServerlessSpec
# 0. pinecone 객체 생성
pc = Pinecone() # api_key=os.getenv('PINECONE_API_KEY')

# 1. 인덱스 설정
index_name = 'tax-index-table'
dimension = len(embedding.embed_query("차원 확인용"))

# 2. 인덱스 생성(해당 인덱스가 없을 경우)
existing_indexes = [index.name for index in pc.list_indexes()]
# print(index_name in existing_indexes)
if index_name not in existing_indexes:
    pc.create_index(
        name=index_name, 
        dimension=dimension,
        metric='cosine',  # 유사도 측정 방식
        spec=ServerlessSpec(
            cloud='aws',    # 무료 플랜은 aws
            region='us-east-1' # 무료플랜은 us-east-1만 가능
        )        
    )
    # 인덱스가 준비될 때까지 대기
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1) # 1초 대기
    print(f'인덱스 {index_name}를 생성하였습니다')
else:
    print(f'인덱스 {index_name}이 이미 존재합니다')
```


### 수업본 코드 셀 8

```python
# 3. 인덱스 정보 확인
print([index.name for index in pc.list_indexes()])
print(pc.describe_index(index_name).status['ready'])
index = pc.Index(index_name)
print(index.describe_index_stats())
```


### 수업본 코드 셀 9

```python
%%time
# 4. 문서를 임베딩해서 클라우드 vector DB(index)에 저장
# 처음 한 번
from pinecone import Pinecone
pc = Pinecone()
# from langchain_pinecone import PineconeVectorStore
# database = PineconeVectorStore.from_documents(
#     documents=documents,
#     embedding=embedding,
#     index_name=index_name
# )
# 업로드한 벡터DB를 load할 때
index_name = 'tax-index-table'
database = PineconeVectorStore(
    embedding=embedding,
    index_name=index_name
)
```


# 2. 답변 생성 전 retrieval 확인

### 수업본 코드 셀 11

```python
query = "연봉이 5천만원인 직장인의 소득세는 얼마인가요?"
retrieved_docs = database.similarity_search(query, k=2) # 기본k값은 4
```


### 수업본 코드 셀 12

```python
retrieved_docs[0]
```


### 수업본 코드 셀 13

```python
retriever = database.as_retriever( 
    search_type='similarity',
    search_kwargs={'k':2}
)
retrieved_docs = retriever.invoke("연봉이 5천만원인 직장인의 소득세는 얼마인가요?")
```


### 수업본 코드 셀 14

```python
retrieved_docs[0]
```


### 수업본 코드 셀 15

```python
retrieved_doc = "\n--\n".join([doc.page_content for doc in retrieved_docs])
# print(retrieved_doc)
```


### 수업본 코드 셀 16

```python
'55조' in retrieved_doc
```


# 3. 답변 생성

### 수업본 코드 셀 18

```python
from openai import OpenAI
client = OpenAI()
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
response = client.chat.completions.create(
    model = "gpt-4o-mini",
    messages=[
        {'role':'system', 'content':'당신은 최고의 한국 소득세법 전문가입니다'},
        {
            'role':'user',
            'content':f"""- [context]를 참고해서 사용자의 질문에 답변해 주세요.
            - [context] : {retrieved_doc}
            - 질문 : {query}"""
        }
    ],
    temperature=0.2
)
```


### 수업본 코드 셀 19

```python
result = response.choices[0].message.content
print(result)
```


### 수업본 코드 셀 20

```python
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model='gpt-4.1-mini')
```


### 수업본 코드 셀 21

```python
from langchain_upstage import ChatUpstage
llm = ChatUpstage(
    model='solar-pro4',
    reasoning_effort='low' # 느리지만 더 깊게 추론하여 답변 (low, medium, high)
)
```


### 수업본 코드 셀 22

```python
prompt = f"""[identity]
- 당신은 최고의 한국 소득세법 전문가입니다.
- [context]를 참고하여 사용자의 질문에 답변해 주세요. [context]는 다음과 같아요
{retrieved_doc}
- 질문 : {query}"""
```


### 수업본 코드 셀 23

```python
ai_message = llm.invoke(prompt)
print(ai_message.content)
```


## Langchain

### 수업본 코드 셀 25

```python
# from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_upstage import ChatUpstage, UpstageEmbeddings
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
# 1. 임베딩 & LLM 초기화
load_dotenv()
# embedding = OpenAIEmbeddings(model = "text-embedding-3-large")
# llm = ChatOpenAI(model='gpt-4.1-mini')
embedding = UpstageEmbeddings(model='embedding-query')
llm = ChatUpstage(model='solar-pro4')

# 2. vector DB load
# pc = Pinecone()
index_name = 'tax-index-table'
vectorStore = PineconeVectorStore(
    embedding=embedding,
    index_name=index_name
)

# 3. Retriever 생성 - search_type='similarity' : 가장 유사한 top-k개 추출
retriever = vectorStore.as_retriever(search_type='similarity', 
                                     search_kwargs={'k':2}) # 유사도가 높은 top-2추출

# 4. 프롬프트템플릿
template = f"""당신은 최고의 한국 소득세법 전문가입니다. 다음 문맥을 참고하여 질문에 답하세요. 답을 모르면 모른다고 
답하세요. 최대 3문장으로 간결하게 답변하세요
질문 : {{query}}
문맥 : {{context}}
답변 :"""
prompt = ChatPromptTemplate.from_template(template)
# 5. 검색된 문서를 텍스트로 변환하는 함수
def format_documents(retrieved_docs):
    return "\n--\n".join([doc.page_content.replace('\n\n', ' ') for doc in retrieved_docs])
```


### 수업본 코드 셀 26

```python
# 6. RAG 렝체인 구성 (LCEL방식)
# 6. RAG 체인 구성
rag_chain = (
    {
        "context":retriever | format_documents,
        "query":RunnablePassthrough() # 질문만 그대로 전달
    }
    | prompt
    | llm
    | StrOutputParser()
)
result = rag_chain.invoke('연봉 5천만원인 직장인의 소득세는 얼마인가요?')
print(result)
```
